# Download GSE190564 to the activation-liability Drive

This notebook downloads the official processed GSE190564 archive and SOFT metadata into:

[https://drive.google.com/drive/folders/11hz8VhGV2bcSGp3dWmeQlufIsIeZQAzF](https://drive.google.com/drive/folders/11hz8VhGV2bcSGp3dWmeQlufIsIeZQAzF)

It performs resumable downloads, SHA-256 hashing, TAR validation, GEX/ADT inventory and optional
95 MB splitting. Successful download does **not** enable the dataset or change any scientific claim.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/projects_bioinfo_2026/activation-liability-real-data")
DEST = DRIVE_ROOT / "06_TISSUE_PROTEIN_VALIDATION" / "GSE190564_UC_PAIRED_CITESEQ"
DEST.mkdir(parents=True, exist_ok=True)

ARCHIVE_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE190nnn/GSE190564/suppl/GSE190564_processed_data.tar.gz"
SOFT_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE190nnn/GSE190564/soft/GSE190564_family.soft.gz"
)
ARCHIVE = DEST / "GSE190564_processed_data.tar.gz"
SOFT = DEST / "GSE190564_family.soft.gz"
CREATE_SPLIT_COPY = True
PART_SIZE_BYTES = 95_000_000
print(DEST)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y aria2

In [ ]:
import subprocess


def aria2_download(url: str, output: Path) -> None:
    command = [
        "aria2c",
        "--continue=true",
        "--max-connection-per-server=8",
        "--split=8",
        "--min-split-size=16M",
        "--file-allocation=none",
        f"--dir={output.parent}",
        f"--out={output.name}",
        url,
    ]
    subprocess.run(command, check=True)


aria2_download(ARCHIVE_URL, ARCHIVE)
aria2_download(SOFT_URL, SOFT)
print(f"Archive: {ARCHIVE.stat().st_size:,} bytes")
print(f"SOFT:    {SOFT.stat().st_size:,} bytes")

In [ ]:
import gzip
import hashlib
import json
import tarfile
from datetime import UTC, datetime


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


assert ARCHIVE.stat().st_size >= 10_000_000_000, "Archive is unexpectedly small"
with gzip.open(SOFT, "rt", encoding="utf-8", errors="replace") as handle:
    assert "GSE190564" in handle.read(4096)

with tarfile.open(ARCHIVE, "r:gz") as tar:
    members = [m for m in tar.getmembers() if m.isfile()]

names = [m.name for m in members]
gex = [n for n in names if "GEX" in n.upper()]
adt = [n for n in names if "ADT" in n.upper()]
assert gex and adt, "Both GEX and ADT members are required"

manifest = {
    "accession": "GSE190564",
    "status": "DOWNLOAD_VALIDATED_NOT_ANALYSED",
    "created_utc": datetime.now(UTC).isoformat(),
    "files": [
        {
            "name": ARCHIVE.name,
            "size_bytes": ARCHIVE.stat().st_size,
            "sha256": sha256_file(ARCHIVE),
        },
        {"name": SOFT.name, "size_bytes": SOFT.stat().st_size, "sha256": sha256_file(SOFT)},
    ],
    "archive_member_count": len(names),
    "gex_member_count": len(gex),
    "adt_member_count": len(adt),
}
(DEST / "download_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
manifest

In [ ]:
import csv

with (DEST / "archive_inventory.tsv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle, delimiter="\t")
    writer.writerow(["member_name", "size_bytes", "modality"])
    for member in members:
        upper = member.name.upper()
        modality = "ADT" if "ADT" in upper else ("GEX" if "GEX" in upper else "OTHER")
        writer.writerow([member.name, member.size, modality])

print("GEX members:", len(gex))
print("ADT members:", len(adt))
print("Inventory:", DEST / "archive_inventory.tsv")

In [ ]:
import re

pool_pattern = re.compile(r"(POOL[^/_.-]*)", re.IGNORECASE)
rows = []
for name in names:
    match = pool_pattern.search(name)
    pool = match.group(1) if match else "UNRESOLVED"
    upper = name.upper()
    modality = "ADT" if "ADT" in upper else ("GEX" if "GEX" in upper else "OTHER")
    rows.append((pool, modality, name))
with (DEST / "pool_inventory.tsv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle, delimiter="\t")
    writer.writerow(["pool_candidate", "modality", "member_name"])
    writer.writerows(rows)
print("Pool inventory written. Patient mapping still requires metadata verification.")

In [ ]:
if CREATE_SPLIT_COPY:
    split_dir = DEST / "split"
    split_dir.mkdir(exist_ok=True)
    parts = []
    with ARCHIVE.open("rb") as source:
        index = 1
        while True:
            chunk = source.read(PART_SIZE_BYTES)
            if not chunk:
                break
            part = split_dir / f"{ARCHIVE.name}.part{index:03d}"
            if not part.exists() or part.stat().st_size != len(chunk):
                part.write_bytes(chunk)
            parts.append(
                {
                    "index": index,
                    "name": part.name,
                    "size_bytes": part.stat().st_size,
                    "sha256": sha256_file(part),
                }
            )
            index += 1
    split_manifest = {
        "source_name": ARCHIVE.name,
        "source_size_bytes": ARCHIVE.stat().st_size,
        "source_sha256": sha256_file(ARCHIVE),
        "part_size_bytes": PART_SIZE_BYTES,
        "parts": parts,
        "reassembly": "Concatenate parts in numeric filename order.",
    }
    (split_dir / f"{ARCHIVE.name}.split-manifest.json").write_text(
        json.dumps(split_manifest, indent=2, sort_keys=True) + "\n"
    )
    print(f"Created {len(parts)} verified parts in {split_dir}")

## Stop here

The download is complete when `download_manifest.json`, `archive_inventory.tsv` and
`pool_inventory.tsv` exist. Do not inspect target effects or change scoring rules in this notebook.
The next phase will verify patient/condition hashtags, GEX–ADT alignment and the antibody panel.
